# Case 4 LIFTS Example

Train LIFTS on synthetic Case 4 data with encoder calibration enabled.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "lifts").exists():
        sys.path.insert(0, str(candidate / "src"))
        break

from lifts import LIFTSImputer, make_case4_dataset, plot_probabilistic_imputation

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

p, d, T = 20, 5, 48
obs_rate = 0.5

data = make_case4_dataset(
    num_sample_train=200,
    num_sample_test=50,
    p=p,
    d=d,
    T=T,
    obs_rate=obs_rate,
    seed=SEED,
    mm="mcar",
    version="v1",
)

print("X_train:", data["X_train"].shape)
print("mask_train:", data["mask_train"].shape)
print("X_test:", data["X_test"].shape)

In [ ]:
model = LIFTSImputer(
    p=p,
    T=T,
    d=d,
    obs_rate=obs_rate,
    total_loops=20,
    batch_size=64,
    n_samples=20,
    lambda_selfmask=5.0,
    lambda_enc=1.0,
    enc_warmup=5,
    seed=SEED,
)

model.fit(data["X_train"], data["mask_train"], verbose=True)

## Impute And Plot

In [ ]:
samples = model.impute_samples(data["X_test"], data["mask_test"], n_samples=20)
print("samples:", samples.shape)

example_index = 0
plot_probabilistic_imputation(
    torch.from_numpy(samples[example_index]),
    torch.from_numpy(data["X_test_full"][example_index]),
    p=p,
    T=T,
    obs_mask=torch.from_numpy(data["mask_test"][example_index]),
    title="LIFTS Case 4 probabilistic imputation",
)